# MHA, MQA, and GQA from Scratch

Multi-Head Attention (MHA), Multi-Query Attention (MQA), and Grouped-Query Attention (GQA): the attention variants that trade quality for inference efficiency.

**Interview questions:**
- "Why MQA/GQA?"
- "What is the quality vs efficiency tradeoff?"
- "How do you convert MHA to GQA?"
- "How does this affect the KV cache?"

---

## Key Idea

| Variant | Q heads | K heads | V heads | KV-cache per token |
|---------|---------|---------|---------|-------------------|
| MHA | H | H | H | $2 \times H \times d_k$ |
| MQA | H | **1** | **1** | $2 \times 1 \times d_k$ |
| GQA-G | H | **G** | **G** | $2 \times G \times d_k$ |

where $H$ = number of heads, $G$ = number of groups, $d_k$ = head dimension.

GQA interpolates between MHA (G=H) and MQA (G=1). LLaMA 2 70B, Gemini, and Mistral all use GQA.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Standard Multi-Head Attention (MHA)

Each head has its own Q, K, V projections. This is the original Transformer design.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Standard Multi-Head Attention: separate Q, K, V per head."""
    
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Separate projections for Q, K, V (each has n_heads sets of d_k dimensions)
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # (d_model, n_heads * d_k)
        self.W_k = nn.Linear(d_model, d_model, bias=False)  # (d_model, n_heads * d_k)
        self.W_v = nn.Linear(d_model, d_model, bias=False)  # (d_model, n_heads * d_k)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x: torch.Tensor, kv_cache: dict = None,
                use_cache: bool = False) -> tuple:
        """Forward pass with optional KV cache.
        
        Args:
            x: (batch, seq_len, d_model)
            kv_cache: dict with 'k' and 'v' tensors from previous steps
            use_cache: whether to return updated cache
        """
        B, T, _ = x.shape
        
        # Project Q, K, V
        q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, T, d_k)
        k = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, T, d_k)
        v = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, T, d_k)
        
        # Handle KV cache for autoregressive inference
        if kv_cache is not None:
            k = torch.cat([kv_cache['k'], k], dim=2)
            v = torch.cat([kv_cache['v'], v], dim=2)
        
        new_cache = {'k': k, 'v': v} if use_cache else None
        
        # Attention
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, H, T, S)
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, self.d_model)
        
        return self.W_o(out), new_cache
    
    @property
    def kv_cache_size_per_token(self) -> int:
        """Number of elements in KV cache per token."""
        return 2 * self.n_heads * self.d_k  # K + V, each (n_heads, d_k)


# Test
d_model, n_heads = 512, 8
mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(2, 32, d_model)
out, _ = mha(x)

print(f"MHA: d_model={d_model}, n_heads={n_heads}, d_k={d_model//n_heads}")
print(f"Params: {sum(p.numel() for p in mha.parameters()):,}")
print(f"KV cache per token: {mha.kv_cache_size_per_token} elements")
print(f"Output shape: {out.shape}")

## 2. Multi-Query Attention (MQA)

Shazeer (2019): all heads share a **single** set of K and V projections. Only Q varies per head.

This reduces the KV cache by a factor of $H$ (number of heads), which is the main bottleneck during autoregressive inference.

In [ ]:
class MultiQueryAttention(nn.Module):
    """Multi-Query Attention: shared K, V across all heads.
    
    Q: n_heads separate projections (d_model -> n_heads * d_k)
    K: single shared projection (d_model -> d_k)
    V: single shared projection (d_model -> d_k)
    """
    
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)       # H separate Q projections
        self.W_k = nn.Linear(d_model, self.d_k, bias=False)      # 1 shared K projection
        self.W_v = nn.Linear(d_model, self.d_k, bias=False)      # 1 shared V projection
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x: torch.Tensor, kv_cache: dict = None,
                use_cache: bool = False) -> tuple:
        B, T, _ = x.shape
        
        # Q: per-head projections
        q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, T, d_k)
        
        # K, V: single shared projection
        k = self.W_k(x).unsqueeze(1)  # (B, 1, T, d_k) -- broadcast across heads
        v = self.W_v(x).unsqueeze(1)  # (B, 1, T, d_k)
        
        # KV cache
        if kv_cache is not None:
            k = torch.cat([kv_cache['k'], k], dim=2)
            v = torch.cat([kv_cache['v'], v], dim=2)
        
        new_cache = {'k': k, 'v': v} if use_cache else None
        
        # Attention: k,v broadcast across heads
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, H, T, S)
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, self.d_model)
        
        return self.W_o(out), new_cache
    
    @property
    def kv_cache_size_per_token(self) -> int:
        return 2 * 1 * self.d_k  # only 1 K and 1 V head


mqa = MultiQueryAttention(d_model, n_heads)
out_mqa, _ = mqa(x)

print(f"MQA: d_model={d_model}, n_heads={n_heads}, d_k={d_model//n_heads}")
print(f"Params: {sum(p.numel() for p in mqa.parameters()):,}")
print(f"KV cache per token: {mqa.kv_cache_size_per_token} elements")
print(f"Output shape: {out_mqa.shape}")

## 3. Grouped-Query Attention (GQA)

Ainslie et al. (2023): compromise between MHA and MQA. Heads are divided into $G$ groups, each group shares one set of K,V.

- GQA-1 = MQA (1 group)
- GQA-H = MHA (H groups, each with 1 head)

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention: groups of heads share K,V projections.
    
    n_heads must be divisible by n_kv_heads.
    n_kv_heads = 1 -> MQA
    n_kv_heads = n_heads -> MHA
    """
    
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups = n_heads // n_kv_heads  # heads per KV group
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x: torch.Tensor, kv_cache: dict = None,
                use_cache: bool = False) -> tuple:
        B, T, _ = x.shape
        
        # Q projections: all heads
        q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        # (B, n_heads, T, d_k)
        
        # K,V projections: only n_kv_heads
        k = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        # (B, n_kv_heads, T, d_k)
        
        # KV cache
        if kv_cache is not None:
            k = torch.cat([kv_cache['k'], k], dim=2)
            v = torch.cat([kv_cache['v'], v], dim=2)
        
        new_cache = {'k': k, 'v': v} if use_cache else None
        
        # Repeat K,V for each group of Q heads
        # (B, n_kv_heads, S, d_k) -> (B, n_heads, S, d_k)
        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)
        
        # Standard attention
        S = k.shape[2]  # total sequence length (including cache)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, self.d_model)
        
        return self.W_o(out), new_cache
    
    @property
    def kv_cache_size_per_token(self) -> int:
        return 2 * self.n_kv_heads * self.d_k


# Test GQA with different group sizes
for n_kv in [1, 2, 4, 8]:
    gqa = GroupedQueryAttention(d_model, n_heads=8, n_kv_heads=n_kv)
    out_gqa, _ = gqa(x)
    params = sum(p.numel() for p in gqa.parameters())
    variant = "MQA" if n_kv == 1 else ("MHA" if n_kv == 8 else f"GQA-{n_kv}")
    print(f"{variant:>6s} (kv_heads={n_kv}): params={params:>8,}, "
          f"KV-cache/token={gqa.kv_cache_size_per_token:>4d} elements, "
          f"output={out_gqa.shape}")

## 4. Head Structure Visualization

In [ ]:
def visualize_attention_heads(n_heads=8, n_kv_heads_list=[8, 4, 2, 1]):
    """Visualize Q/K/V head assignments for different attention variants."""
    fig, axes = plt.subplots(1, len(n_kv_heads_list), figsize=(4*len(n_kv_heads_list), 6))
    
    colors = plt.cm.Set3(np.linspace(0, 1, n_heads))
    
    for idx, n_kv in enumerate(n_kv_heads_list):
        ax = axes[idx]
        heads_per_group = n_heads // n_kv
        
        variant = "MHA" if n_kv == n_heads else ("MQA" if n_kv == 1 else f"GQA-{n_kv}")
        ax.set_title(f"{variant}\n(kv_heads={n_kv})", fontsize=12, fontweight='bold')
        
        # Draw Q heads
        for i in range(n_heads):
            group = i // heads_per_group
            color = colors[group * heads_per_group]  # color by group
            rect = plt.Rectangle((0.1, n_heads - 1 - i), 0.8, 0.8,
                               facecolor=color, edgecolor='black', linewidth=0.5)
            ax.add_patch(rect)
            ax.text(0.5, n_heads - 0.6 - i, f'Q{i}', ha='center', va='center', fontsize=8)
        
        # Draw K heads
        for i in range(n_kv):
            color = colors[i * heads_per_group]
            height = heads_per_group * 0.8 + (heads_per_group - 1) * 0.2
            y_start = n_heads - 1 - i * heads_per_group - (heads_per_group - 1)
            rect = plt.Rectangle((1.5, y_start), 0.8, height,
                               facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            ax.text(1.9, y_start + height/2, f'K{i}', ha='center', va='center', fontsize=8)
        
        # Draw V heads
        for i in range(n_kv):
            color = colors[i * heads_per_group]
            height = heads_per_group * 0.8 + (heads_per_group - 1) * 0.2
            y_start = n_heads - 1 - i * heads_per_group - (heads_per_group - 1)
            rect = plt.Rectangle((2.9, y_start), 0.8, height,
                               facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            ax.text(3.3, y_start + height/2, f'V{i}', ha='center', va='center', fontsize=8)
        
        # Draw connections
        for i in range(n_heads):
            group = i // heads_per_group
            q_y = n_heads - 0.6 - i
            kv_y_start = n_heads - 1 - group * heads_per_group - (heads_per_group - 1)
            kv_y = kv_y_start + (heads_per_group * 0.8 + (heads_per_group-1)*0.2) / 2
            ax.plot([0.9, 1.5], [q_y, kv_y], 'gray', linewidth=0.5, alpha=0.5)
        
        ax.set_xlim(-0.2, 4.2)
        ax.set_ylim(-0.5, n_heads + 0.5)
        ax.set_xticks([0.5, 1.9, 3.3])
        ax.set_xticklabels(['Q', 'K', 'V'], fontsize=11)
        ax.set_yticks([])
    
    plt.suptitle('Attention Head Structures', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

visualize_attention_heads(n_heads=8, n_kv_heads_list=[8, 4, 2, 1])

## 5. KV-Cache Size Comparison

During autoregressive generation, the KV cache stores K and V for all previous tokens. This is the main memory bottleneck.

In [ ]:
# KV cache size for different model configurations
# Inspired by real models

configs = {
    'GPT-3 175B (MHA)':       {'d_model': 12288, 'n_heads': 96, 'n_kv_heads': 96, 'n_layers': 96},
    'LLaMA-2 70B (GQA-8)':    {'d_model': 8192,  'n_heads': 64, 'n_kv_heads': 8,  'n_layers': 80},
    'Mistral 7B (GQA-8)':     {'d_model': 4096,  'n_heads': 32, 'n_kv_heads': 8,  'n_layers': 32},
    'Falcon 40B (MQA)':       {'d_model': 8192,  'n_heads': 64, 'n_kv_heads': 1,  'n_layers': 60},
}

seq_len = 4096  # context length
batch_size = 1
dtype_bytes = 2  # fp16

print(f"KV Cache Size for seq_len={seq_len}, batch_size={batch_size}, FP16:")
print("=" * 80)

cache_sizes = {}
for name, cfg in configs.items():
    d_k = cfg['d_model'] // cfg['n_heads']
    # Cache per layer: 2 (K+V) * n_kv_heads * seq_len * d_k * dtype_bytes
    per_layer = 2 * cfg['n_kv_heads'] * seq_len * d_k * dtype_bytes
    total = per_layer * cfg['n_layers']
    
    # What it would be with MHA
    per_layer_mha = 2 * cfg['n_heads'] * seq_len * d_k * dtype_bytes
    total_mha = per_layer_mha * cfg['n_layers']
    
    cache_sizes[name] = total / 1e9
    
    reduction = total_mha / total if total > 0 else 1
    print(f"{name:>30s}: {total/1e9:>6.2f} GB  "
          f"(vs MHA: {total_mha/1e9:.2f} GB, {reduction:.0f}x reduction)")

print("=" * 80)

In [ ]:
# Visualize KV cache scaling with sequence length
seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768]
d_model = 4096
n_heads = 32
n_layers = 32
d_k = d_model // n_heads

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

variants = {
    'MHA (32 KV heads)': 32,
    'GQA-8 (8 KV heads)': 8,
    'GQA-4 (4 KV heads)': 4,
    'MQA (1 KV head)': 1,
}

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for (name, n_kv), color in zip(variants.items(), colors):
    cache_gb = [2 * n_kv * s * d_k * n_layers * 2 / 1e9 for s in seq_lengths]
    axes[0].plot(seq_lengths, cache_gb, '-o', label=name, color=color, markersize=5)

axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('KV Cache Size (GB)')
axes[0].set_title(f'KV Cache Size vs Sequence Length\n(d={d_model}, H={n_heads}, L={n_layers}, FP16)')
axes[0].legend()
axes[0].set_xscale('log', base=2)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=24, color='gray', linestyle='--', alpha=0.5)
axes[0].text(600, 25, 'RTX 3090 (24GB)', fontsize=8, color='gray')

# Parameter count comparison
kv_heads_list = [1, 2, 4, 8, 16, 32]
kv_params = []
total_attn_params = []

for n_kv in kv_heads_list:
    # Q: d_model * n_heads * d_k
    q_params = d_model * n_heads * d_k
    # K,V: d_model * n_kv * d_k each
    k_params = d_model * n_kv * d_k
    v_params = d_model * n_kv * d_k
    # O: d_model * d_model
    o_params = d_model * d_model
    
    kv_params.append((k_params + v_params) * n_layers)
    total_attn_params.append((q_params + k_params + v_params + o_params) * n_layers)

x_pos = np.arange(len(kv_heads_list))
axes[1].bar(x_pos, [p/1e6 for p in total_attn_params], color='steelblue', edgecolor='white')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'{n}\n({"MQA" if n==1 else ("MHA" if n==32 else f"GQA-{n}")})'
                         for n in kv_heads_list])
axes[1].set_xlabel('Number of KV Heads')
axes[1].set_ylabel('Attention Parameters (M)')
axes[1].set_title('Attention Layer Parameters')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Throughput Comparison (Autoregressive Decoding)

In [ ]:
def measure_decoding_throughput(attn_module, d_model, n_steps=50, warmup=10):
    """Measure autoregressive decoding throughput with KV cache."""
    attn_module.eval()
    
    # Warmup
    cache = None
    x = torch.randn(1, 1, d_model)
    for _ in range(warmup):
        out, cache = attn_module(x, kv_cache=cache, use_cache=True)
    
    # Reset cache
    cache = None
    
    # Measure
    start = time.time()
    for _ in range(n_steps):
        x = torch.randn(1, 1, d_model)
        with torch.no_grad():
            out, cache = attn_module(x, kv_cache=cache, use_cache=True)
    elapsed = time.time() - start
    
    tokens_per_sec = n_steps / elapsed
    return tokens_per_sec, cache


d_model = 256
n_heads = 8

variants = {
    'MHA': MultiHeadAttention(d_model, n_heads),
    'MQA': MultiQueryAttention(d_model, n_heads),
    'GQA-2': GroupedQueryAttention(d_model, n_heads, n_kv_heads=2),
    'GQA-4': GroupedQueryAttention(d_model, n_heads, n_kv_heads=4),
}

results = {}
for name, module in variants.items():
    tps, cache = measure_decoding_throughput(module, d_model, n_steps=100)
    cache_elements = sum(v.numel() for v in cache.values())
    results[name] = {'tps': tps, 'cache_size': cache_elements}
    print(f"{name:>6s}: {tps:.0f} tokens/sec, KV cache={cache_elements:,} elements")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = list(results.keys())
tps_values = [results[n]['tps'] for n in names]
cache_values = [results[n]['cache_size'] for n in names]

axes[0].bar(names, tps_values, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'], edgecolor='white')
axes[0].set_ylabel('Tokens/sec')
axes[0].set_title('Decoding Throughput')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(names, cache_values, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'], edgecolor='white')
axes[1].set_ylabel('KV Cache Elements')
axes[1].set_title('KV Cache Size After 100 Tokens')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Converting MHA to GQA (Uptraining)

The LLaMA 2 paper (Touvron et al., 2023) showed that you can convert a pre-trained MHA model to GQA by **mean-pooling** the K,V heads within each group, then continuing training.

This is more efficient than training GQA from scratch.

In [ ]:
def convert_mha_to_gqa(mha: MultiHeadAttention, n_kv_heads: int) -> GroupedQueryAttention:
    """Convert a pre-trained MHA module to GQA by mean-pooling K,V heads.
    
    Within each group, average the K and V weight matrices.
    Q weights and O weights are kept as-is.
    """
    d_model = mha.d_model
    n_heads = mha.n_heads
    d_k = mha.d_k
    
    assert n_heads % n_kv_heads == 0
    heads_per_group = n_heads // n_kv_heads
    
    gqa = GroupedQueryAttention(d_model, n_heads, n_kv_heads)
    
    # Copy Q and O weights directly
    gqa.W_q.weight.data = mha.W_q.weight.data.clone()
    gqa.W_o.weight.data = mha.W_o.weight.data.clone()
    
    # Mean-pool K weights: (n_heads, d_k, d_model) -> (n_kv_heads, d_k, d_model)
    k_weight = mha.W_k.weight.data.view(n_heads, d_k, d_model)
    k_grouped = k_weight.view(n_kv_heads, heads_per_group, d_k, d_model).mean(dim=1)
    gqa.W_k.weight.data = k_grouped.view(n_kv_heads * d_k, d_model)
    
    # Mean-pool V weights
    v_weight = mha.W_v.weight.data.view(n_heads, d_k, d_model)
    v_grouped = v_weight.view(n_kv_heads, heads_per_group, d_k, d_model).mean(dim=1)
    gqa.W_v.weight.data = v_grouped.view(n_kv_heads * d_k, d_model)
    
    return gqa


# Demo conversion
mha = MultiHeadAttention(512, 8)
x = torch.randn(2, 16, 512)

mha_out, _ = mha(x)

# Convert to GQA-4 and GQA-2
for n_kv in [4, 2, 1]:
    gqa = convert_mha_to_gqa(mha, n_kv_heads=n_kv)
    gqa_out, _ = gqa(x)
    
    diff = (mha_out - gqa_out).abs().mean().item()
    mha_params = sum(p.numel() for p in mha.parameters())
    gqa_params = sum(p.numel() for p in gqa.parameters())
    
    variant = "MQA" if n_kv == 1 else f"GQA-{n_kv}"
    print(f"MHA -> {variant}: mean output diff = {diff:.4f}, "
          f"param reduction = {(1 - gqa_params/mha_params)*100:.1f}%")

print("\nNote: output differs because K,V are averaged. Quality recovers with continued training (uptraining).")

## 8. Batch Size and Memory Analysis

The KV cache advantage of MQA/GQA grows with **batch size** -- serving many users simultaneously.

In [ ]:
# How many concurrent users can we serve?
gpu_memory_gb = 80  # A100
model_memory_gb = 14  # 7B model in FP16
available_for_cache = gpu_memory_gb - model_memory_gb

d_model = 4096
n_heads = 32
n_layers = 32
d_k = d_model // n_heads
seq_len = 4096
dtype_bytes = 2  # FP16

variants_batch = {
    'MHA (32 KV heads)': 32,
    'GQA-8 (8 KV heads)': 8,
    'GQA-4 (4 KV heads)': 4,
    'MQA (1 KV head)': 1,
}

max_batches = {}
for name, n_kv in variants_batch.items():
    cache_per_request_gb = 2 * n_kv * d_k * seq_len * n_layers * dtype_bytes / 1e9
    max_batch = int(available_for_cache / cache_per_request_gb)
    max_batches[name] = max_batch
    print(f"{name:>25s}: {cache_per_request_gb:.2f} GB/request -> max batch = {max_batch}")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(max_batches.keys(), max_batches.values(),
              color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'], edgecolor='white')

for bar, v in zip(bars, max_batches.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(v), ha='center', fontweight='bold', fontsize=12)

ax.set_ylabel('Max Concurrent Requests')
ax.set_title(f'Max Batch Size on A100 80GB (7B model, seq_len={seq_len})')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 9. Quality Comparison (Synthetic Task)

In [ ]:
# Train small models with different attention types on a simple task

class SmallLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_kv_heads, n_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(128, d_model)
        self.attns = nn.ModuleList([
            GroupedQueryAttention(d_model, n_heads, n_kv_heads) for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        h = self.embed(x) + self.pos(torch.arange(T, device=x.device))
        for attn, norm in zip(self.attns, self.norms):
            res = h
            h = norm(h)
            h, _ = attn(h)
            h = h + res
        return self.head(h)

# Simple copy task
vocab_size = 100
seq_len = 32
data = torch.randint(0, vocab_size, (500, seq_len))

configs_train = {
    'MHA (8/8)': 8,
    'GQA (8/4)': 4,
    'GQA (8/2)': 2,
    'MQA (8/1)': 1,
}

all_losses = {}
for name, n_kv in configs_train.items():
    model = SmallLM(vocab_size, d_model=128, n_heads=8, n_kv_heads=n_kv, n_layers=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    losses = []
    for epoch in range(20):
        perm = torch.randperm(len(data))
        epoch_loss = 0
        n = 0
        for i in range(0, len(data), 32):
            batch = data[perm[i:i+32]]
            logits = model(batch[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), batch[:, 1:].reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n += 1
        losses.append(epoch_loss / n)
    
    all_losses[name] = losses
    params = sum(p.numel() for p in model.parameters())
    print(f"{name}: final_loss={losses[-1]:.4f}, params={params:,}")

plt.figure(figsize=(10, 5))
for name, losses in all_losses.items():
    plt.plot(losses, '-o', label=name, markersize=3)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss: MHA vs GQA vs MQA')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Models Using GQA/MQA in Practice

| Model | Variant | n_heads | n_kv_heads | Reason |
|-------|---------|---------|------------|--------|
| GPT-3 | MHA | 96 | 96 | Pre-dates GQA |
| PaLM | MQA | 16 | 1 | Early MQA adoption |
| Falcon 40B | MQA | 64 | 1 | Maximum KV cache efficiency |
| LLaMA-2 7B | MHA | 32 | 32 | Small model, MHA is fine |
| LLaMA-2 70B | GQA-8 | 64 | 8 | Large model needs KV savings |
| Mistral 7B | GQA-8 | 32 | 8 | GQA even for smaller models |
| Gemma | MQA | varies | 1 | Google preference |
| Gemini | GQA | varies | varies | Balances quality and speed |

In [ ]:
# Final summary table
print("=" * 80)
print(f"{'Variant':>10} {'Q heads':>10} {'KV heads':>10} {'KV cache':>15} {'Quality':>10} {'Speed':>10}")
print("=" * 80)
print(f"{'MHA':>10} {'H':>10} {'H':>10} {'2*H*d_k*L':>15} {'Best':>10} {'Slowest':>10}")
print(f"{'GQA-G':>10} {'H':>10} {'G':>10} {'2*G*d_k*L':>15} {'Near MHA':>10} {'Fast':>10}")
print(f"{'MQA':>10} {'H':>10} {'1':>10} {'2*1*d_k*L':>15} {'Slightly <':>10} {'Fastest':>10}")
print("=" * 80)
print("\nH = total heads, G = KV groups, d_k = head dim, L = num layers")
print("\nGQA with G=8 is the current industry standard for models >13B parameters.")

## Interview Notes

**"Why MQA/GQA?"**
- During autoregressive decoding, KV cache is the main memory bottleneck
- KV cache grows linearly with sequence length and batch size
- MQA reduces KV cache by H times (e.g., 32x for 32-head model)
- GQA provides a middle ground: G times reduction with minimal quality loss
- More concurrent users = higher throughput = lower serving cost

**"Quality vs efficiency tradeoff?"**
- MQA can degrade quality slightly (1-2% on benchmarks) due to reduced K,V capacity
- GQA-8 matches MHA quality within noise on most benchmarks (LLaMA 2 paper)
- The quality gap narrows for larger models (more total capacity)
- For serving-heavy workloads, the throughput gain far outweighs the tiny quality loss

**"How to convert MHA to GQA?"**
- Mean-pool K,V heads within each group
- Keep Q and output projection weights unchanged
- Continue pre-training ("uptrain") for a fraction of the original training budget
- LLaMA 2: uptrained for only ~5% of original compute
- This is much cheaper than training GQA from scratch

**"How does this relate to the KV cache?"**
- In MHA: cache stores $K, V \in \mathbb{R}^{n\_heads \times seq\_len \times d_k}$ per layer
- In GQA: cache stores $K, V \in \mathbb{R}^{n\_kv\_heads \times seq\_len \times d_k}$ per layer
- At serving time, K,V are broadcast/repeated to match Q heads
- This broadcast is cheap compute but the cache savings are massive for memory

**Key numbers to know:**
- LLaMA-2 70B: 64 Q heads, 8 KV heads (8x cache reduction)
- Mistral 7B: 32 Q heads, 8 KV heads
- GQA-8 is the sweet spot for most large models